# Batch Runner Notebook

Run the batch pipeline in order: `collector -> deduper -> structurer`.
Edit the parameters below before executing.

In [1]:
import os
import re
import sys
import subprocess
from pathlib import Path
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        if (path / "backend").is_dir() and (path / "frontend").is_dir():
            return path
    return start


ROOT = find_repo_root(Path.cwd())
BACKEND_ROOT = ROOT / "backend"

# Load .env from repo root (if present)
env_path = ROOT / ".env"
load_dotenv(env_path if env_path.exists() else None)

# Ensure backend is on sys.path for "app.*" imports in later cells
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

# Parameters
MUNICIPALITY_ID = "tokyo-chiyoda"
MUNICIPALITY_NAME = "Chiyoda"
DOMAIN = "childcare"  # "moving" or "childcare"

# Optional: set if you already have a catalog id; otherwise it will be created by the collector
CATALOG_ID = ""


BATCH_MAIN = BACKEND_ROOT / "app" / "batch" / "main.py"


def run_batch(job: str, catalog_id: str | None = None) -> str | None:
    cmd = [
        sys.executable,
        str(BATCH_MAIN),
        "--job", job,
        "--municipality_id", MUNICIPALITY_ID,
        "--domain", DOMAIN,
    ]
    if MUNICIPALITY_NAME:
        cmd += ["--municipality_name", MUNICIPALITY_NAME]
    if catalog_id:
        cmd += ["--catalog_id", catalog_id]

    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    new_id = None
    if job == "collector":
        m = re.search(r"Created new Catalog:\s*([a-f0-9-]+)", result.stdout)
        if m:
            new_id = m.group(1)
            print("Detected catalog_id:", new_id)
    return new_id


## 1) Collector
Creates a new catalog if `CATALOG_ID` is empty.

In [ ]:
if CATALOG_ID:
    run_batch("collector", CATALOG_ID)
else:
    new_id = run_batch("collector")
    if new_id:
        CATALOG_ID = new_id

print("CATALOG_ID:", CATALOG_ID)


## 2) Deduper

In [ ]:
if not CATALOG_ID:
    raise ValueError("CATALOG_ID is required. Run collector first or set it manually.")

run_batch("deduper", CATALOG_ID)


## 3) Structurer

In [ ]:
if not CATALOG_ID:
    raise ValueError("CATALOG_ID is required. Run collector first or set it manually.")

run_batch("structurer", CATALOG_ID)


## 4) Export CSV
Exports programs from Firestore to a CSV file using `backend/scripts/export_csv.py`.

In [ ]:
# CSV output location
OUTPUT_CSV = f"programs_{CATALOG_ID}.csv"

if not CATALOG_ID:
    raise ValueError("CATALOG_ID is required. Run collector first or set it manually.")

cmd = [
    sys.executable,
    str(ROOT / "backend" / "scripts" / "export_csv.py"),
    CATALOG_ID,
    OUTPUT_CSV,
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CSV saved to:", OUTPUT_CSV)


## 5) Pipeline (collector -> deduper -> structurer)
Runs all targets in targets.json.


In [2]:
cmd = [
    sys.executable,
    str(BATCH_MAIN),
    "--job", "pipeline",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)


Running: /home/keita/.pyenv/versions/3.13.9/bin/python /mnt/c/Users/nakan/Desktop/work/competitions/Gov-Alternate/backend/app/batch/main.py --job pipeline
Created new Catalog: cf4dc1cc-f129-427b-9535-65fb2e40a7c2
Created new Catalog: db363385-36d2-47c0-ac61-331706e7d520
Created new Catalog: 70f78e74-dbb5-44d0-a717-5d567a59f8c5
Created new Catalog: a592ca96-70c3-4e72-be0f-4dcbf75214f0
Created new Catalog: 697014b6-dafa-4c47-a01f-8a8f12d9cf76
Created new Catalog: 47f5a90c-de4e-4d3d-afcc-144aba45a00d
Created new Catalog: 6ba2705d-0291-4e01-b87e-cb2e9281d226
Error fetching https://www.pref.saitama.lg.jp/documents/9099/daityo_202512.xlsx: Client error '404 Not Found' for url 'https://www.pref.saitama.lg.jp/documents/9099/daityo_202512.xlsx'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
Error fetching https://www.pref.saitama.lg.jp/documents/9099/daityo_202512.xlsx: Client error '404 Not Found' for url 'https://www.pref.saitama.lg.jp/documents/9099/

## Debug: Vertex AI Search logs
Enable verbose logs for SearchService and run pipeline from notebook.


In [ ]:
import os

# Enable debug logs for Vertex AI Search
os.environ["VERTEX_AI_SEARCH_DEBUG"] = "1"
print("VERTEX_AI_SEARCH_DEBUG=", os.environ.get("VERTEX_AI_SEARCH_DEBUG"))

cmd = [
    sys.executable,
    str(BATCH_MAIN),
    "--job", "pipeline",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)


## Debug: Single Search Query
Run a single query against Vertex AI Search via SearchService and print parsed URLs.


In [ ]:
import json
from pathlib import Path

from app.services.search import SearchService
from app.batch.targets import DEFAULT_TARGETS_FILE, load_targets_file, resolve_engine_ids

# Adjust query/municipality as needed
query = "千代田区 転入届"
municipality_id = "tokyo-chiyoda"

targets = load_targets_file(Path(DEFAULT_TARGETS_FILE))
target = next(t for t in targets["targets"] if t["municipality_id"] == municipality_id)
search_municipality_ids = target.get("search_municipality_ids") or [municipality_id]
engine_ids = resolve_engine_ids(targets, search_municipality_ids)

svc = SearchService()
results = svc.execute_search(query, num=5, engine_ids=engine_ids)

print("engine_ids=", engine_ids)
print("results=", len(results))
for r in results:
    print("-", r.get("url"), "|", r.get("title"))
